# EDA 1단계 — 데이터 개요 파악

**목적**: 데이터가 어떻게 생겼는지 파악
- 행(row) / 열(column) 수 확인
- 각 컬럼의 자료형 확인
- 스케일 문제 확인 (기상청 원본값 → 실제값 변환 필요 여부)

**분석 대상 데이터**
- `master_grid.parquet` : 격자 기본 정보 (지형, 임상, 전신주)
- `grid_date_master/` : 날씨 데이터 (2025년 2월~5월, 월별 파티션)

---
## 0. 라이브러리 로드

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 출력 설정: 컬럼 잘리지 않게
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('라이브러리 로드 완료')

라이브러리 로드 완료


---
## 1. 데이터 불러오기

In [2]:
BASE_PATH = r'D:\prj_2_공모전\data\003. 전처리 데이터(preprocessing_data)'

# 격자 마스터 (지형·임상·전신주 정보)
master = pd.read_parquet(f'{BASE_PATH}\\master_grid.parquet')

# 날씨 데이터 (2025년 2~5월 전체)
weather = pd.read_parquet(f'{BASE_PATH}\\grid_date_master')

print(f'master_grid   : {master.shape[0]:,}행, {master.shape[1]}열')
print(f'grid_date_master : {weather.shape[0]:,}행, {weather.shape[1]}열')

master_grid   : 273,001행, 18열
grid_date_master : 32,760,120행, 19열


---
## 2. master_grid 개요

강원도 전체를 100m×100m 격자로 나눈 '지도 기준표'입니다.  
> 각 격자마다 전신주 개수, 산림 여부, 지형 정보가 담겨 있어요.

In [3]:
print('=== master_grid 기본 정보 ===')
print(f'\n행(격자 수): {master.shape[0]:,}개')
print(f'열(변수 수): {master.shape[1]}개')
print(f'\n메모리 사용량: {master.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

=== master_grid 기본 정보 ===

행(격자 수): 273,001개
열(변수 수): 18개

메모리 사용량: 135.6 MB


In [4]:
print('=== 컬럼별 자료형 및 결측 현황 ===')

# 자료형 + 결측치 수 + 결측 비율 한 번에
master_info = pd.DataFrame({
    '자료형': master.dtypes,
    '결측 수': master.isnull().sum(),
    '결측률(%)': (master.isnull().sum() / len(master) * 100).round(2),
    '고유값 수': master.nunique()
})

print(master_info.to_string())

=== 컬럼별 자료형 및 결측 현황 ===
                         자료형   결측 수  결측률(%)   고유값 수
grid_id               object      0  0.0000  273001
grid_x                 int32      0  0.0000    1648
grid_y                 int32      0  0.0000    1322
pole_count             int64      0  0.0000      42
is_forest              int32      0  0.0000       2
forest_exist_code     object  67329 24.6600       3
forest_origin_code    object  67329 24.6600       3
forest_type_code      object  67329 24.6600       5
tree_species          object  67329 24.6600      36
diameter_class_code   object  73523 26.9300       4
age_class_code        object  73523 26.9300       9
density_code          object  73523 26.9300       3
tree_height_code      object  73533 26.9400      20
forest_updated_year   object  88229 32.3200      11
elevation            float64      0  0.0000     306
slope                float64      0  0.0000   30781
aspect_sin           float64      0  0.0000   56979
aspect_cos           float64      0  0.0

In [5]:
print('=== master_grid 샘플 (상위 3행) ===')
master.head(3)

=== master_grid 샘플 (상위 3행) ===


,grid_id,grid_x,grid_y,pole_count,is_forest,forest_exist_code,forest_origin_code,forest_type_code,tree_species,diameter_class_code,age_class_code,density_code,tree_height_code,forest_updated_year,elevation,slope,aspect_sin,aspect_cos
0,10007_19696,10007,19696,1,0,None,None,None,None,None,None,None,None,None,50.0000,2.8624,-1.0000,0.0000
1,10008_19691,10008,19691,15,0,None,None,None,None,None,None,None,None,None,55.0000,55.3476,0.7242,-0.6896
2,10008_19692,10008,19692,2,0,None,None,None,None,None,None,None,None,None,60.0000,55.0114,0.6983,0.7158


In [6]:
print('=== 수치형 컬럼 기본 통계 ===')
master.describe().T

=== 수치형 컬럼 기본 통계 ===


,count,mean,std,min,25%,50%,75%,max
grid_x,273001.0000,10791.4376,405.7627,10007.0000,10417.0000,10815.0000,11128.0000,11654.0000
grid_y,273001.0000,19517.1873,288.5634,18939.0000,19276.0000,19504.0000,19732.0000,20261.0000
pole_count,273001.0000,5.0836,3.3006,1.0000,3.0000,4.0000,7.0000,60.0000
is_forest,273001.0000,0.7534,0.4310,0.0000,1.0000,1.0000,1.0000,1.0000
elevation,273001.0000,441.5036,374.2161,0.0000,135.0000,330.0000,680.0000,1570.0000
slope,273001.0000,32.2994,25.9629,0.0000,4.5202,33.1773,56.4687,84.2075
aspect_sin,273001.0000,0.0061,0.6673,-1.0000,-0.7071,0.0000,0.7071,1.0000
aspect_cos,273001.0000,0.1176,0.7354,-1.0000,-0.7071,0.0000,0.9378,1.0000


---
## 3. weather(grid_date_master) 개요

각 격자마다 날짜별 날씨 정보가 담긴 데이터입니다.  
> '격자 수 × 날짜 수'만큼 행이 있어야 정상입니다.  
> **주의**: 기상청 원본이라 기온·습도 등의 숫자가 실제값보다 10배 클 수 있습니다.

In [7]:
print('=== weather 기본 정보 ===')
print(f'\n행(격자×날짜): {weather.shape[0]:,}개')
print(f'열(변수 수): {weather.shape[1]}개')
print(f'\n메모리 사용량: {weather.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

print(f'\n날짜 범위: {weather["date"].min()} ~ {weather["date"].max()}')
print(f'날짜 수: {weather["date"].nunique()}일')
print(f'고유 격자 수: {weather["grid_id"].nunique():,}개')
print(f'\n이론상 예상 행 수: {weather["date"].nunique()} × {weather["grid_id"].nunique():,} = {weather["date"].nunique() * weather["grid_id"].nunique():,}')
print(f'실제 행 수: {weather.shape[0]:,}')

=== weather 기본 정보 ===

행(격자×날짜): 32,760,120개
열(변수 수): 19개

메모리 사용량: 6154.8 MB

날짜 범위: 2025-02-01 00:00:00 ~ 2025-05-31 00:00:00
날짜 수: 120일
고유 격자 수: 273,001개

이론상 예상 행 수: 120 × 273,001 = 32,760,120
실제 행 수: 32,760,120


In [8]:
print('=== 컬럼별 자료형 및 결측 현황 ===')

weather_info = pd.DataFrame({
    '자료형': weather.dtypes,
    '결측 수': weather.isnull().sum(),
    '결측률(%)': (weather.isnull().sum() / len(weather) * 100).round(2),
    '고유값 수': weather.nunique()
})

print(weather_info.to_string())

=== 컬럼별 자료형 및 결측 현황 ===
                             자료형  결측 수  결측률(%)    고유값 수
grid_id                   object     0  0.0000   273001
kma_nx                     int64     0  0.0000      325
kma_ny                     int64     0  0.0000      258
ta_mean                  float64     0  0.0000     3664
ta_max                   float64     0  0.0000      483
hm_mean                  float64     0  0.0000     6859
hm_min                   float64     0  0.0000      958
td_mean                  float64     0  0.0000     3626
td_min                   float64     0  0.0000      500
wind_ws_mean             float64     0  0.0000      684
wind_ws_max              float64     0  0.0000      203
wind_uu_mean             float64     0  0.0000     1020
wind_vv_mean             float64     0  0.0000     1082
wind_wd_sin_mean         float64     0  0.0000  3413999
wind_wd_cos_mean         float64     0  0.0000  3414000
rn_day_mean              float64     0  0.0000     1745
rn_day_max              

In [9]:
print('=== weather 샘플 (상위 3행) ===')
weather.head(3)

=== weather 샘플 (상위 3행) ===


,grid_id,kma_nx,kma_ny,ta_mean,ta_max,hm_mean,hm_min,td_mean,td_min,wind_ws_mean,wind_ws_max,wind_uu_mean,wind_vv_mean,wind_wd_sin_mean,wind_wd_cos_mean,rn_day_mean,rn_day_max,date,month
0,10007_19696,1139,1483,-9.3750,59.0000,881.7500,631.0000,-29.3750,-43.0000,3.7500,6.0000,-1.7500,-0.2500,0.4841,0.2262,0.3750,1.0000,2025-02-01,2025-02
1,10008_19691,1139,1482,-10.7500,58.0000,879.8750,632.0000,-30.2500,-44.0000,3.3750,6.0000,-1.5000,-0.3750,0.2521,0.0236,0.3750,1.0000,2025-02-01,2025-02
2,10008_19692,1139,1482,-10.7500,58.0000,879.8750,632.0000,-30.2500,-44.0000,3.3750,6.0000,-1.5000,-0.3750,0.2521,0.0236,0.3750,1.0000,2025-02-01,2025-02


---
## 4. ★ 핵심: 스케일(단위) 문제 확인
  
> 기상청 원본 데이터는 저장 공간을 줄이기 위해 소수점 없이 정수로 저장합니다.  
> 예: 실제 기온 12.5°C → 저장값 125 (÷10 해야 실제값)  
>  
> 아래에서 **실제 값의 범위**가 상식적으로 맞는지 확인합니다.

| 변수 | 현실적인 범위 | 스케일 적용 전 | 의심 기준 |
|------|------------|-------------|----------|
| 기온 (ta) | -20 ~ 40°C | -200 ~ 400 | 절대값 > 100이면 ÷10 필요 |
| 습도 (hm) | 0 ~ 100% | 0 ~ 1000 | 최대값 > 100이면 ÷10 필요 |
| 이슬점 (td) | -30 ~ 35°C | -300 ~ 350 | 절대값 > 100이면 ÷10 필요 |
| 강수량 (rn) | 0 ~ 200mm | 0 ~ 2000 | 최대값 > 500이면 ÷10 의심 |
| 풍속 (wind_ws) | 0 ~ 50m/s | 확인 필요 | - |

In [10]:
# 스케일 검증 대상 컬럼
scale_check_cols = [
    'ta_mean', 'ta_max',
    'hm_mean', 'hm_min',
    'td_mean', 'td_min',
    'wind_ws_mean', 'wind_ws_max',
    'wind_uu_mean', 'wind_vv_mean',
    'rn_day_mean', 'rn_day_max'
]

# 현재 데이터의 실제 범위 확인
scale_df = pd.DataFrame({
    '최솟값': weather[scale_check_cols].min(),
    '평균값': weather[scale_check_cols].mean().round(2),
    '최댓값': weather[scale_check_cols].max(),
    '÷10 후 최솟값': (weather[scale_check_cols].min() / 10).round(2),
    '÷10 후 평균값': (weather[scale_check_cols].mean() / 10).round(2),
    '÷10 후 최댓값': (weather[scale_check_cols].max() / 10).round(2),
})

print('=== 스케일 검증 (현재값 vs ÷10 후 값) ===')
print(scale_df.to_string())

=== 스케일 검증 (현재값 vs ÷10 후 값) ===
                   최솟값      평균값       최댓값  ÷10 후 최솟값  ÷10 후 평균값  ÷10 후 최댓값
ta_mean      -193.6250  69.3700  271.6250   -19.3600     6.9400    27.1600
ta_max       -162.0000 123.1600  321.0000   -16.2000    12.3200    32.1000
hm_mean       140.7500 621.3400 1000.0000    14.0800    62.1300   100.0000
hm_min         43.0000 404.7200 1000.0000     4.3000    40.4700   100.0000
td_mean      -257.7500  -9.6800  207.6250   -25.7800    -0.9700    20.7600
td_min       -342.0000 -40.0600  204.0000   -34.2000    -4.0100    20.4000
wind_ws_mean    0.5000  13.3300   89.2500     0.0500     1.3300     8.9300
wind_ws_max     1.0000  24.5100  235.0000     0.1000     2.4500    23.5000
wind_uu_mean  -58.7500   5.7400   82.2500    -5.8800     0.5700     8.2200
wind_vv_mean  -79.0000   3.1500   81.6250    -7.9000     0.3200     8.1600
rn_day_mean     0.0000   5.8000  222.3750     0.0000     0.5800    22.2400
rn_day_max      0.0000  17.0900  550.0000     0.0000     1.7100    5

In [11]:
print('=== 스케일 판단 기준 적용 ===')

# 각 변수의 상식적 최대 범위 (스케일 적용 전 원본 기준)
# 원본이 올바르다면 이 범위 안에 있어야 함
expected_raw = {
    'ta_mean':     (-40, 50),    # 기온 °C
    'ta_max':      (-40, 50),
    'hm_mean':     (0, 100),     # 습도 %
    'hm_min':      (0, 100),
    'td_mean':     (-40, 40),    # 이슬점 °C
    'td_min':      (-40, 40),
    'wind_ws_mean':(0, 60),      # 풍속 m/s
    'wind_ws_max': (0, 100),
    'wind_uu_mean':(-60, 60),    # U성분 풍속
    'wind_vv_mean':(-60, 60),
    'rn_day_mean': (0, 500),     # 강수량 mm
    'rn_day_max':  (0, 1000),
}

results = []
for col, (lo, hi) in expected_raw.items():
    actual_min = weather[col].min()
    actual_max = weather[col].max()
    in_range = (actual_min >= lo) and (actual_max <= hi)
    results.append({
        '컬럼': col,
        '현재 최솟값': actual_min,
        '현재 최댓값': actual_max,
        f'기대 범위': f'{lo} ~ {hi}',
        '스케일 OK?': '✅ 정상' if in_range else '⚠️ 변환 필요 의심'
    })

result_df = pd.DataFrame(results).set_index('컬럼')
print(result_df.to_string())

=== 스케일 판단 기준 적용 ===
                현재 최솟값    현재 최댓값     기대 범위      스케일 OK?
컬럼                                                     
ta_mean      -193.6250  271.6250  -40 ~ 50  ⚠️ 변환 필요 의심
ta_max       -162.0000  321.0000  -40 ~ 50  ⚠️ 변환 필요 의심
hm_mean       140.7500 1000.0000   0 ~ 100  ⚠️ 변환 필요 의심
hm_min         43.0000 1000.0000   0 ~ 100  ⚠️ 변환 필요 의심
td_mean      -257.7500  207.6250  -40 ~ 40  ⚠️ 변환 필요 의심
td_min       -342.0000  204.0000  -40 ~ 40  ⚠️ 변환 필요 의심
wind_ws_mean    0.5000   89.2500    0 ~ 60  ⚠️ 변환 필요 의심
wind_ws_max     1.0000  235.0000   0 ~ 100  ⚠️ 변환 필요 의심
wind_uu_mean  -58.7500   82.2500  -60 ~ 60  ⚠️ 변환 필요 의심
wind_vv_mean  -79.0000   81.6250  -60 ~ 60  ⚠️ 변환 필요 의심
rn_day_mean     0.0000  222.3750   0 ~ 500         ✅ 정상
rn_day_max      0.0000  550.0000  0 ~ 1000         ✅ 정상


---
## 5. 월별 날짜 및 격자 수 분포 확인

2월~5월 4개월 동안 데이터가 고르게 있는지 확인합니다.

In [12]:
# date 타입 변환
weather['date'] = pd.to_datetime(weather['date'])
weather['month'] = weather['date'].dt.to_period('M')

monthly = weather.groupby('month').agg(
    날짜수=('date', 'nunique'),
    격자수=('grid_id', 'nunique'),
    행수=('grid_id', 'count')
).reset_index()

print('=== 월별 데이터 현황 ===')
print(monthly.to_string(index=False))

=== 월별 데이터 현황 ===
  month  날짜수    격자수      행수
2025-02   28 273001 7644028
2025-03   31 273001 8463031
2025-04   30 273001 8190030
2025-05   31 273001 8463031


---
## 6. 1단계 결과 요약

> 아래 셀을 실행하면 핵심 결과를 한눈에 정리해서 보여줍니다.

In [13]:
print('=' * 55)
print('  1단계 데이터 개요 요약')
print('=' * 55)

print(f'''
[master_grid]
  - 격자(행) 수   : {master.shape[0]:,}개
  - 변수(열) 수   : {master.shape[1]}개
  - 메모리        : {master.memory_usage(deep=True).sum()/1024**2:.1f} MB

[grid_date_master (날씨)]
  - 행 수         : {weather.shape[0]:,}개
  - 변수(열) 수   : {weather.shape[1]}개
  - 기간          : {weather["date"].min().date()} ~ {weather["date"].max().date()}
  - 날짜 수       : {weather["date"].nunique()}일
  - 격자 수       : {weather["grid_id"].nunique():,}개
  - 메모리        : {weather.memory_usage(deep=True).sum()/1024**2:.1f} MB
''')

print('[스케일 검증 결과]')
for r in results:
    print(f"  {r['컬럼']:<20}: {r['스케일 OK?']}  (현재값 범위: {r['현재 최솟값']:.1f} ~ {r['현재 최댓값']:.1f})")

print()
print('→ 다음 단계: 2단계 결측치·이상치 분석으로 이동')
print('=' * 55)

  1단계 데이터 개요 요약

[master_grid]
  - 격자(행) 수   : 273,001개
  - 변수(열) 수   : 18개
  - 메모리        : 135.6 MB

[grid_date_master (날씨)]
  - 행 수         : 32,760,120개
  - 변수(열) 수   : 19개
  - 기간          : 2025-02-01 ~ 2025-05-31
  - 날짜 수       : 120일
  - 격자 수       : 273,001개
  - 메모리        : 6373.5 MB

[스케일 검증 결과]
  ta_mean             : ⚠️ 변환 필요 의심  (현재값 범위: -193.6 ~ 271.6)
  ta_max              : ⚠️ 변환 필요 의심  (현재값 범위: -162.0 ~ 321.0)
  hm_mean             : ⚠️ 변환 필요 의심  (현재값 범위: 140.8 ~ 1000.0)
  hm_min              : ⚠️ 변환 필요 의심  (현재값 범위: 43.0 ~ 1000.0)
  td_mean             : ⚠️ 변환 필요 의심  (현재값 범위: -257.8 ~ 207.6)
  td_min              : ⚠️ 변환 필요 의심  (현재값 범위: -342.0 ~ 204.0)
  wind_ws_mean        : ⚠️ 변환 필요 의심  (현재값 범위: 0.5 ~ 89.2)
  wind_ws_max         : ⚠️ 변환 필요 의심  (현재값 범위: 1.0 ~ 235.0)
  wind_uu_mean        : ⚠️ 변환 필요 의심  (현재값 범위: -58.8 ~ 82.2)
  wind_vv_mean        : ⚠️ 변환 필요 의심  (현재값 범위: -79.0 ~ 81.6)
  rn_day_mean         : ✅ 정상  (현재값 범위: 0.0 ~ 222.4)
  rn_day_max          : ✅ 정상  (현재